# 03 — Impact Ionization Coefficients  


---
## 1. Imports

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.utils.ingestion import DataIngestionConfig, DataIngestionService
from src.avalanche.ionization import (
    VanOverstraetenDeManModel, OkutoCrowellModel,
    IonizationCoefficients
)
from src.core.constants import q, kB

print('Imports OK')

---
## 2. Van Overstraeten–de Man model (Default)

Two-regime Chynoweth form with phonon-scattering temperature dependence:

$$
\alpha(E) = 
\begin{cases}
A_{\text{low}} \exp\!\left(-\dfrac{B_{\text{low}}}{E}\right) & E < E_0 \\[8pt]
A_{\text{high}} \exp\!\left(-\dfrac{B_{\text{high}}}{E}\right) & E \ge E_0
\end{cases}
$$

Temperature enters through $\gamma(T) = \tanh(\hbar\omega/2kT_{\text{ref}}) / \tanh(\hbar\omega/2kT)$:  
$A(T) = A(T_{\text{ref}}) \cdot \gamma(T)$, $B(T) = B(T_{\text{ref}}) \cdot \gamma(T)$.

In [ ]:
vodm = VanOverstraetenDeManModel()
T_ref = 300.0

E_field = np.logspace(4.5, 6.2, 100)  # V/cm

# Get coefficients at 300 K and 350 K
alpha_300 = np.array([vodm.alpha(e, None, 300.0) for e in E_field])
beta_300  = np.array([vodm.beta(e, None, 300.0) for e in E_field])
alpha_350 = np.array([vodm.alpha(e, None, 350.0) for e in E_field])
beta_350  = np.array([vodm.beta(e, None, 350.0) for e in E_field])

plt.figure(figsize=(10, 4))
plt.loglog(E_field, alpha_300, 'b-', lw=2, label='$\\alpha$ (300 K)')
plt.loglog(E_field, beta_300, 'b--', lw=2, label='$\\beta$ (300 K)')
plt.loglog(E_field, alpha_350, 'r-', lw=1.5, label='$\\alpha$ (350 K)')
plt.loglog(E_field, beta_350, 'r--', lw=1.5, label='$\\beta$ (350 K)')
plt.axvline(3.85e5, color='gray', ls=':', alpha=0.5, label='$E_0$ (regime switch)')
plt.xlabel('Electric Field (V/cm)')
plt.ylabel('Ionization Coefficient (cm⁻¹)')
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.title('Van Overstraeten–de Man: α and β vs Field')
plt.show()

### Temperature dependence

At higher T, $\gamma(T) > 1$, so $B$ increases and $\alpha$ decreases. This is why $V_{BR}$ **increases** with temperature — the field must be higher to overcome the reduced ionization efficiency.

In [ ]:
temps = np.arange(250, 400, 10)
E_test = 4.5e5  # V/cm (typical peak field)

gamma_vals = []
for T in temps:
    hw_op_eV = 0.063
    hw_J = hw_op_eV * float(q.magnitude)
    s = np.tanh(hw_J / (2.0 * float(kB.magnitude) * T))
    s_ref = np.tanh(hw_J / (2.0 * float(kB.magnitude) * 300.0))
    gamma_vals.append(s_ref / s)

plt.figure(figsize=(8, 3))
plt.plot(temps, gamma_vals, 'o-', lw=2)
plt.axhline(1, color='gray', ls='--', alpha=0.4)
plt.xlabel('Temperature (K)')
plt.ylabel('$\\gamma(T)$')
plt.grid(alpha=0.3)
plt.title('Optical-phonon scaling factor $\\gamma(T)$')
plt.show()

---
## 3. Okuto-Crowell model

An alternative model with a different functional form:

$$
\alpha(E) = \frac{qE}{E_{\text{th}}} \cdot \exp\!\left[S - \sqrt{S^2 + \left(\frac{E_{\text{th}}}{qE\lambda}\right)^2}\right], \quad S = 0.217\, (E_{\text{th}} / E_R)^{1.14}
$$

In [ ]:
oc = OkutoCrowellModel()

# We need a Material object for Okuto-Crowell (it reads params from material data)
config = DataIngestionConfig.from_defaults()
svc = DataIngestionService(config)
device = svc.build_device()
inp = device.materials['InP']

alpha_oc_300 = np.array([oc.alpha(e, inp, 300.0) for e in E_field])
beta_oc_300  = np.array([oc.beta(e, inp, 300.0) for e in E_field])

plt.figure(figsize=(10, 4))
plt.loglog(E_field, alpha_300, 'b-', lw=1.5, label='VODM $\\alpha$ (300 K)')
plt.loglog(E_field, beta_300, 'b--', lw=1.5, label='VODM $\\beta$ (300 K)')
plt.loglog(E_field, alpha_oc_300, 'r-', lw=1.5, label='OC $\\alpha$ (300 K)')
plt.loglog(E_field, beta_oc_300, 'r--', lw=1.5, label='OC $\\beta$ (300 K)')
plt.xlabel('Electric Field (V/cm)')
plt.ylabel('Ionization Coefficient (cm⁻¹)')
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.title('VODM vs Okuto-Crowell: Comparison')
plt.show()

---
## 4. Dead-space correction

In thin multiplication regions, carriers need to travel a minimum distance $l_d$ before they can ionize. This reduces the effective ionization coefficients:

$$\alpha_{\text{eff}} = \frac{\alpha}{1 + \alpha \cdot l_d}, \quad l_d = \frac{E_{\text{th}}}{|E|}$$

In [ ]:
# Load the device ionization model (built during simulation setup)
sim = svc.build_simulator()

phi, E, _ = sim.solve_poisson(55.0)  # near breakdown
E_abs = np.abs(E)
x_um = sim.grid.x * 1e4

alpha_raw = sim.ionization.alpha_n(E_abs)
beta_raw  = sim.ionization.alpha_p(E_abs)

alpha_eff = sim.ionization.effective_alpha_n(E_abs, Eg=1.35)
beta_eff  = sim.ionization.effective_alpha_p(E_abs, Eg=1.35)

# Dead space length
l_e = sim.ionization.dead_space_length(E, 'electron', Eg=1.35)
l_h = sim.ionization.dead_space_length(E, 'hole', Eg=1.35)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.semilogy(x_um, alpha_raw, 'b-', lw=1.5, label='$\\alpha$ (raw)')
ax1.semilogy(x_um, alpha_eff, 'b--', lw=2, label='$\\alpha_{\\text{eff}}$ (dead-space)')
ax1.semilogy(x_um, beta_raw, 'r-', lw=1.5, label='$\\beta$ (raw)')
ax1.semilogy(x_um, beta_eff, 'r--', lw=2, label='$\\beta_{\\text{eff}}$ (dead-space)')
ax1.set_xlabel('Depth (µm)')
ax1.set_ylabel('Ionization coeff (cm⁻¹)')
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)
ax1.set_title('Ionization coeffs at V = 55 V')

ax2.plot(x_um, l_e * 1e4, 'b-', lw=2, label='$l_d$ electron (µm)')
ax2.plot(x_um, l_h * 1e4, 'r-', lw=2, label='$l_d$ hole (µm)')
ax2.set_xlabel('Depth (µm)')
ax2.set_ylabel('Dead space length (µm)')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)
ax2.set_title('Dead-space length across device')

plt.tight_layout()
plt.show()

---
## 5. Ionization ratio $k = \beta/\alpha$ and excess noise

The ratio $k = \beta/\alpha$ determines the **excess noise factor**:

$$F(M) = kM + (1-k)(2 - 1/M)$$

A low $k$ (e.g., 0.02 for InP) means cleaner multiplication with less noise.

In [ ]:
# Ionization ratio vs field
k_raw = beta_300 / (alpha_300 + 1e-30)

plt.figure(figsize=(8, 3))
plt.semilogx(E_field, k_raw, 'g-', lw=2)
plt.xlabel('Electric Field (V/cm)')
plt.ylabel('$k = \\beta/\\alpha$')
plt.grid(alpha=0.3)
plt.title('Ionization ratio vs field (VODM, InP, 300 K)')
plt.axhline(0.02, color='gray', ls='--', alpha=0.4, label='k ~ 0.02 in Geiger range')
plt.legend()
plt.show()

In [ ]:
# Excess noise factor F(M) for different k values
M_range = np.logspace(0, 3, 100)
for k in [0.01, 0.02, 0.05, 0.1, 0.5]:
    F = k * M_range + (1-k) * (2 - 1/M_range)
    plt.loglog(M_range, F, lw=2, label=f'k = {k}')
plt.xlabel('Multiplication M')
plt.ylabel('Excess Noise Factor F(M)')
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.title('McIntyre excess noise factor')
plt.show()

---
## 6. Summary

- **Van Overstraeten–de Man** is the default model — two-regime Chynoweth with optical-phonon temperature scaling.
- **Okuto-Crowell** is an alternative with a different functional form (parameterized in materials XML).
- Higher **temperature reduces ionization** → $V_{BR}$ rises ($dV_{BR}/dT > 0$).
- **Dead-space correction** suppresses $\alpha$ in thin layers by requiring carriers to travel $l_d$ before ionizing.
- InP has $k \approx 0.02$ in the operating range — excellent for low-noise APDs/SPADs.

**Next experiment (04):** Dark current components — SRH, BTBT, TAT — and how they determine the dark count rate.